# AI Risk Prediction Framework - Exploratory Analysis & Model Demonstration

This notebook provides an interactive workspace to explore the AI Risk Prediction models, visualize data distributions, run SHAP explainability analyses, and simulate scenario overrides (What-If Analysis).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure the root of the project is in python path
sys.path.insert(0, os.path.abspath(".."))

from src.config import Paths
print("Configured Data Path:", Paths.ML_READY_DATA)
print("Configured Model Path:", Paths.XGB_MODEL)

## 1. Load and Inspect Dataset

We load the compiled real-world dataset `data/ml_ready_data.csv` which normalizes issues extracted from Apache JIRA, GitHub, and BugSwarm build databases.

In [ ]:
df = pd.read_csv(Paths.ML_READY_DATA)
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 2. Visualize Target Variables and Features

Let's check the class distribution of our risk target variable `Risk_Level` and visualize the relationship between actual days and risk levels.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="Risk_Level", order=["Low", "Medium", "High"], palette="viridis")
plt.title("Distribution of Ticket Risk Levels")
plt.xlabel("Risk Level")
plt.ylabel("Ticket Count")
plt.show()

# Show numeric distributions
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="Risk_Level", y="Actual_Days", palette="mako")
plt.title("Actual Days to Resolution by Risk Level")
plt.xlabel("Risk Level")
plt.ylabel("Actual Days")
plt.show()

## 3. Load Trained Model & Preprocess Features

Now let's load our production classifier `models/xgb_model.pkl`. We'll print details of the model, preprocess features to match the model training space, and view column layouts.

In [ ]:
model = joblib.load(Paths.XGB_MODEL)
print("Model type:", type(model).__name__)

from src.preprocessing.feature_engineering import preprocess_features
X = preprocess_features(df)
print("Preprocessed features shape:", X.shape)

## 4. Run SHAP Interpretability Analysis

We'll extract the underlying Booster, run SHAP `TreeExplainer`, and visualize feature contributions for a sample of rows.

In [ ]:
import shap
# Extract booster and patch base_score if needed
booster = model.get_booster()
from src.xai.shap_explainer import patch_booster_base_score
patch_booster_base_score(booster)

# Initialize explainer
explainer = shap.TreeExplainer(booster)
X_sample = X.sample(n=min(50, len(X)), random_state=42)
shap_values = explainer.shap_values(X_sample.values.astype(float))

# Plot SHAP summary (targeting class 2: High Risk if multiclass)
n_classes = shap_values.shape[2] if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3 else len(shap_values)
target_class = 2 if n_classes > 2 else n_classes - 1

print(f"Plotting SHAP Summary for class {target_class} (High Risk)")
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values[:, :, target_class] if shap_values.ndim == 3 else shap_values[target_class],
    X_sample.values.astype(float),
    feature_names=list(X.columns),
    show=False
)
plt.show()

## 5. What-If Simulation Engine Demonstration

Let's instantiate the simulation engine and construct a hypothetical scenario override. We will check the impact of extending the timeline or modifying resource constraints on the risk probability.

In [ ]:
from src.simulation.what_if_simulator import WhatIfSimulator

simulator = WhatIfSimulator(model=model, dataset=df)

# Take a sample ticket
sample_ticket = df.iloc[0]
print("Baseline Ticket details:")
print(f"ID: {sample_ticket.get('id')}")
print(f"Priority: {sample_ticket.get('Priority')}")
print(f"Assignee Seniority: {sample_ticket.get('Assignee_Seniority')}")
print(f"Estimated Days: {sample_ticket.get('Estimated_Days')}")
print(f"Budget Allocated: {sample_ticket.get('Budget_Allocated')}")

# Run simulation deltas
# Example: Extend timeline by 5 days, decrease team efficiency to 80%, change Priority to High
deltas = {
    "timeline_extension_days": 5.0,
    "team_efficiency": 0.8,
    "priority_override": "High",
    "timeline_changed": True
}

simulated_ticket = simulator.apply_deltas(sample_ticket, deltas)
comparison = simulator.compare_scenarios(sample_ticket, simulated_ticket)

print("\n--- Simulation Comparison ---")
print(f"Original Risk: {comparison['original']['risk_label']} (High Risk Prob: {comparison['original']['high_risk_pct']:.2f}%)")
print(f"Original Drivers: {', '.join(comparison['original']['top_drivers'])}")
print(f"Simulated Risk: {comparison['simulated']['risk_label']} (High Risk Prob: {comparison['simulated']['high_risk_pct']:.2f}%)")
print(f"Simulated Drivers: {', '.join(comparison['simulated']['top_drivers'])}")
print("\nDeltas:")
print(f"  High Risk Probability Change: {comparison['delta']['high_risk_pct_change']:.2f}%")
print(f"  Timeline Change: {comparison['delta']['timeline_delta']} days")
print(f"  Budget Change: ${comparison['delta']['budget_delta']:.2f}")